# Predicting Cereal Ratings from Nutrition Values
Compare a **classic ML model** (Linear Regression) with a **neural network** (scikit-learn `MLPRegressor`) for predicting a cereal's `rating`.

- **Features (X):** protein, fat, sodium, fiber, carbo, sugars
- **Target (y):** rating
- `name`, `mfr` and `type` are **not** used as features.

## 1. Imports

In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

pd.set_option("display.width", 120)
pd.set_option("display.max_columns", 20)

FEATURES = ["protein", "fat", "sodium", "fiber", "carbo", "sugars"]
TARGET = "rating"

## 2. Load the data
The second line of the file holds data-type labels (`String, Categorical, ...`), not a cereal, so it is skipped.
Only the selected columns are kept.

In [2]:
df = pd.read_csv("cereal.csv", skiprows=[1])
df = df[["name", "mfr", "type", "calories", "protein", "fat", "sodium",
         "fiber", "carbo", "sugars", TARGET]]

num_cols = [c for c in df.columns if c not in ("name", "mfr", "type")]
df[num_cols] = df[num_cols].apply(pd.to_numeric, errors="coerce")

df.head()

,name,mfr,type,calories,protein,fat,sodium,fiber,carbo,sugars,rating
0,100% Bran,N,C,70,4,1,130,10.0,5.0,6,68.402973
1,100% Natural Bran,Q,C,120,3,5,15,2.0,8.0,8,33.983679
2,All-Bran,K,C,70,4,1,260,9.0,7.0,5,59.425505
3,All-Bran with Extra Fiber,K,C,50,4,0,140,14.0,8.0,0,93.704912
4,Almond Delight,R,C,110,2,2,200,1.0,14.0,8,34.384843


In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 77 entries, 0 to 76
Data columns (total 11 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   name      77 non-null     str    
 1   mfr       77 non-null     str    
 2   type      77 non-null     str    
 3   calories  77 non-null     int64  
 4   protein   77 non-null     int64  
 5   fat       77 non-null     int64  
 6   sodium    77 non-null     int64  
 7   fiber     77 non-null     float64
 8   carbo     77 non-null     float64
 9   sugars    77 non-null     int64  
 10  rating    77 non-null     float64
dtypes: float64(3), int64(5), str(3)
memory usage: 6.7 KB


In [4]:
df.describe().round(2)

,calories,protein,fat,sodium,fiber,carbo,sugars,rating
count,77.00,77.00,77.00,77.00,77.00,77.00,77.00,77.00
mean,106.88,2.55,1.01,159.68,2.15,14.60,6.92,42.67
std,19.48,1.09,1.01,83.83,2.38,4.28,4.44,14.05
min,50.00,1.00,0.00,0.00,0.00,-1.00,-1.00,18.04
25%,100.00,2.00,0.00,130.00,1.00,12.00,3.00,33.17
50%,110.00,3.00,1.00,180.00,2.00,14.00,7.00,40.40
75%,110.00,3.00,2.00,210.00,3.00,17.00,11.00,50.83
max,160.00,6.00,5.00,320.00,14.00,23.00,15.00,93.70


## 3. Missing values
There are no blank cells, but this dataset codes missing nutrition values as **-1** (see the `min` of `carbo` and `sugars` above).
Those are converted to `NaN` before dropping.

In [5]:
print(f"Rows: {len(df)}")
print(f"Missing values (NaN) as loaded: {int(df.isna().sum().sum())}")

df[FEATURES] = df[FEATURES].replace(-1, np.nan)
missing = df.isna().sum()
print("\nMissing values after treating -1 as missing:")
print(missing[missing > 0])
df[df.isna().any(axis=1)][["name"] + FEATURES]

Rows: 77
Missing values (NaN) as loaded: 0

Missing values after treating -1 as missing:
carbo     1
sugars    1
dtype: int64


,name,protein,fat,sodium,fiber,carbo,sugars
57,Quaker Oatmeal,5,2,0,2.7,NaN,NaN


In [6]:
df = df.dropna(how="all")   # drop completely empty rows
df = df.dropna()            # drop any remaining row with a missing value
print(f"Rows after cleaning: {len(df)}")

Rows after cleaning: 76


## 4. Features, target, train/test split (80/20) and scaling

In [7]:
X = df[FEATURES]
y = df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42)

scaler = StandardScaler().fit(X_train)      # fit on TRAIN only
X_train_s = scaler.transform(X_train)
X_test_s = scaler.transform(X_test)

print(f"Train: {X_train.shape}, Test: {X_test.shape}")

Train: (60, 6), Test: (16, 6)


In [8]:
def evaluate(name, model):
    pred = model.predict(X_test_s)
    scores = {
        "model": name,
        "MAE": mean_absolute_error(y_test, pred),
        "RMSE": np.sqrt(mean_squared_error(y_test, pred)),
        "R2": r2_score(y_test, pred),
    }
    return scores, pred

## 5. Classic ML model: Linear Regression

In [9]:
lr = LinearRegression().fit(X_train_s, y_train)
lr_res, lr_pred = evaluate("LinearRegression", lr)

coefs = pd.Series(lr.coef_, index=FEATURES).sort_values(key=abs, ascending=False)
print(f"Intercept: {lr.intercept_:.2f}")
print("Coefficients (change in rating per 1 std of feature):")
coefs.round(2)

Intercept: 43.38
Coefficients (change in rating per 1 std of feature):


sugars    -8.09
fiber      5.77
sodium    -4.71
fat       -3.93
protein    2.13
carbo      0.01
dtype: float64

In [10]:
lr_res

{'model': 'LinearRegression', 'MAE': 1.1531794139325273, 'RMSE': np.float64(1.7744723006939227), 'R2': 0.9854402253246719}

## 6. Neural network: MLPRegressor
Two hidden layers (32 → 16, ReLU), Adam optimizer, small L2 penalty.

In [11]:
nn = MLPRegressor(hidden_layer_sizes=(32, 16), activation="relu",
                  solver="adam", alpha=1e-3, learning_rate_init=0.01,
                  max_iter=5000, random_state=42)
nn.fit(X_train_s, y_train)
nn_res, nn_pred = evaluate("Neural network (MLP 32-16)", nn)

print(f"Iterations: {nn.n_iter_}, final training loss: {nn.loss_:.3f}")
nn_res

Iterations: 2513, final training loss: 0.058


{'model': 'Neural network (MLP 32-16)', 'MAE': 2.2161183670026796, 'RMSE': np.float64(2.745087679895521), 'R2': 0.965155948547141}

## 7. Comparison

In [12]:
results = pd.DataFrame([lr_res, nn_res]).set_index("model").round(3)
results

,MAE,RMSE,R2
model,,,
LinearRegression,1.153,1.774,0.985
Neural network (MLP 32-16),2.216,2.745,0.965


In [13]:
preds = pd.DataFrame({
    "name": df.loc[X_test.index, "name"].values,
    "actual": y_test.values,
    "LinearReg": lr_pred,
    "NN": nn_pred,
}).round(2)
preds["LR_error"] = (preds["LinearReg"] - preds["actual"]).round(2)
preds["NN_error"] = (preds["NN"] - preds["actual"]).round(2)
preds

,name,actual,LinearReg,NN,LR_error,NN_error
0,Almond Delight,34.38,30.89,32.51,-3.49,-1.87
1,Honey Graham Ohs,21.87,22.28,23.07,0.41,1.20
2,Cap'n'Crunch,18.04,18.02,18.91,-0.02,0.87
3,100% Bran,68.40,68.28,65.16,-0.12,-3.24
4,Muesli Raisins; Peaches; & Pecans,34.14,33.06,35.44,-1.08,1.30
5,Multi-Grain Cheerios,40.11,39.85,38.41,-0.26,-1.70
6,Smacks,31.23,29.27,34.26,-1.96,3.03
7,Product 19,41.50,43.32,44.47,1.82,2.97
8,Nutri-grain Wheat,59.64,58.36,59.69,-1.28,0.05
9,Fruitful Bran,41.02,40.99,41.94,-0.03,0.92


## 8. Conclusion
- **Linear Regression wins** on every metric (lower MAE and RMSE, higher R²).
- The rating behaves like a near-linear score of the nutrition values: **sugars** pull it down most, then sodium and fat, while **fiber** and protein push it up. A linear model captures this almost exactly.
- The neural network reaches a very low training loss but does worse on the test set: with only ~60 training rows it **overfits**.
- With just 16 test rows the exact numbers depend on the split; k-fold cross-validation would give a steadier comparison.